# Step 2: Data Cleaning, Wrangling & Pre-Processing #

## Goals: ##
* Clean and wrangle dataset. This could include dropping null values, removing unnecessary columns, removing outliers, and potentially fixing incorrectly formatted data.
* Save this dataframe as a new csv file to be used in the next step.


## Drop unncessary columns ##

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 

df = pd.read_csv("../data/bank_transactions.csv")

I'm dropping `nameOrig` and `nameDest` because they are identifier-like fields rather than behavioral predictors. They could potentially help identify specific accounts, but they do not directly describe the transaction patterns that drive fraud, and keeping them would risk pushing the model toward memorizing account IDs instead of learning generalizable fraud behavior.

I also dropped `isFlaggedFraud` because it is an extremely sparse pre-existing flag and is not a reliable feature for modeling fraud behavior in this dataset. In this data, it is only positive for a tiny number of rows and appears to be inconsistent with the fraud label, so it is better treated as a descriptive artifact than as a useful predictor.


In [9]:
df = df.drop(columns=["nameOrig", "nameDest", "isFlaggedFraud"])

I'm creating time- and balance-based features because the EDA suggests fraud is linked to late-night activity and to specific balance changes around the transaction. The hour feature captures time-of-day effects, while the balance-difference and near-equality flags encode the fraud signature more directly, which should help tree-based models learn rule-like patterns more effectively than relying on raw variables alone.


In [10]:
df["hour"] = df["step"] % 24
df["diff_orig"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
df["diff_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]

df["amt_equals_orig_balance"] = (np.isclose(df["amount"], df["oldbalanceOrg"], rtol=0, atol=1)).astype(int)
df["orig_balance_zero_after"] = (df["newbalanceOrig"] == 0).astype(int)
df["dest_balance_unchanged"] = (np.isclose(df["newbalanceDest"], df["oldbalanceDest"], rtol=0, atol=1)).astype(int)


Because `DEBIT` and `PAYMENT` contain no fraud cases in this dataset, I plan to compare two modeling approaches: one trained on all transaction types and one trained only on `TRANSFER` and `CASH_OUT`. This makes it possible to test whether removing the zero-fraud types improves fraud detection without sacrificing performance on unseen data.


In [6]:
df_all_types = df
df_only_fraud_types = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])]

df_all_types needs dummy encoding for the four types but df_only_fraud_types wouldn't need it since it would have only 2 types.

In [8]:
df_all_types = pd.get_dummies(df, columns=['type'], drop_first=True)
df_all_types.head()

,step,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,False,False,True,False
1,1,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,False,False,True,False
2,1,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,False,False,False,True
3,1,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,True,False,False,False
4,1,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,False,False,True,False


In [5]:
df_only_fraud_types

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
2,1,TRANSFER,181.00,C1305486145,181.00,0.0,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.0,C38997010,21182.00,0.00,1,0
15,1,CASH_OUT,229133.94,C905080434,15325.00,0.0,C476402209,5083.00,51513.44,0,0
19,1,TRANSFER,215310.30,C1670993182,705.00,0.0,C1100439041,22425.00,0.00,0,0
24,1,TRANSFER,311685.89,C1984094095,10835.00,0.0,C932583850,6267.00,2719172.89,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1,0


In [ ]:
df = pd.get_dummies(df, columns=["type"], drop_first=True)

In [ ]:
df.head()


In [ ]:
df.to_csv("../data/cleaned_caishen_bank_transactions.csv", index=False)